# Biomechanical and Multimodal Shopper Behavior Analytics

This experimental notebook combines YOLO11-pose and ByteTrack, projected head
orientation, opportunistic HSEmotion expression estimates, and biomechanical
heuristics to explore shopper behavior around configured shelf polygons.

It produces annotated video exports, in-memory event tables, heatmaps and joint
telemetry. The demonstrations illustrate pipeline behavior; overlay labels and
model scores are not ground truth, calibrated accuracy, verified emotion, product
removal or purchase confirmation. The notebook's legacy payloads require adaptation
to the backend's version 2.0 contracts before production ingestion.

Install the separate notebook environment from the root `requirements.txt` as
explained in the [README](../README.md#notebook-reproduction). Model initialization
can download external weights and sample footage when absent. The notebook does
not start Docker services or write to the application database.

Published exports and animated previews are in the
[README demonstration gallery](../README.md#processed-video-demonstrations).
Videos are linked as files rather than embedded as base64 to keep this notebook
reviewable. Individual facial-crop outputs are omitted from the published notebook;
aggregate charts are retained.


In [10]:
# Environment setup

import os
os.environ["YOLO_VERBOSE"] = "False"

import uuid
import json
import subprocess
from datetime import datetime, timezone, timedelta
from collections import deque
from typing import Dict, List, Tuple, Optional, Any

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, display
from shapely.geometry import Point, Polygon, LineString
from ultralytics import YOLO



def to_json_native(value: Any) -> Any:
    """Convert NumPy values to JSON-native scalars and containers."""
    if isinstance(value, np.generic):
        return to_json_native(value.item())
    if isinstance(value, np.ndarray):
        return to_json_native(value.tolist())
    if isinstance(value, dict):
        return {key: to_json_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_native(item) for item in value]
    return value

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
POSE_MODEL = YOLO("yolo11s-pose.pt")

try:
    from hsemotion_onnx.facial_emotions import HSEmotionRecognizer
    FACE_RECOGNIZER = HSEmotionRecognizer(model_name="enet_b0_8_best_vgaf")
    HSEMOTION_AVAILABLE = True
except Exception:
    FACE_RECOGNIZER = None
    HSEMOTION_AVAILABLE = False

print(f"Device: {DEVICE} | HSEmotion Active: {HSEMOTION_AVAILABLE}")

Device: cuda:0 | HSEmotion Active: True


## Scale-Invariant Biomechanics & Affective Kinematics

To avoid the need for large-scale custom video labeling during initial deployment, we leverage pre-trained skeletal keypoints (`YOLO11-pose`) and apply deterministic **kinematic equations** established in ergonomics and affective computing literature.

### 1. Scale-Invariant Anthropometric Normalization
In oblique CCTV perspectives, pixel distances vary with camera depth. We adopt the projected **Biacromial Width ($W_s$)**—the Euclidean distance between the left ($P_5$) and right ($P_6$) shoulders—as an intrinsic anatomical unit at frame $t$:

$$W_s(t) = \Vert{} P_6(t) - P_5(t) \Vert{}_2 = \sqrt{(x_6 - x_5)^2 + (y_6 - y_5)^2}$$

All linear displacements, reach distances, and inter-hand proximities are normalized by $W_s(t)$, yielding dimensionless ratios invariant to camera distance and shopper height (*Popa et al., 2013*).

---

### 2. Upper-Limb Joint Kinematics (Reaching vs. Product Inspection)
Following the *MERL Shopping Action Primitives* (*Singh et al., 2016*) and the *Rapid Upper Limb Assessment* (*RULA — McAtamney & Corlett, 1993*), shelf interactions are governed by shoulder-elbow coupling:

* **Elbow Flexion/Extension Angle ($\theta_{\text{elbow}}$):** Given the upper-arm vector $\vec{u} = P_{\text{shoulder}} - P_{\text{elbow}}$ and forearm vector $\vec{v} = P_{\text{wrist}} - P_{\text{elbow}}$:
  $$\theta_{\text{elbow}} = \arccos\left( \frac{\vec{u} \cdot \vec{v}}{\Vert{}\vec{u}\Vert{}_2 \Vert{}\vec{v}\Vert{}_2} \right) \cdot \frac{180^\circ}{\pi}$$
  * **Shelf Reach:** $\theta_{\text{elbow}} \in [130^\circ, 180^\circ]$ combined with a normalized reach ratio $R_{\text{reach}} = \frac{\Vert{}P_{\text{wrist}} - P_{\text{shoulder}}\Vert{}_2}{W_s} > 1.10$.
  * **Product Inspection:** $\theta_{\text{elbow}} \in [50^\circ, 115^\circ]$, bringing the item toward the central visual fovea.

* **Shoulder Elevation/Abduction Angle ($\theta_{\text{shoulder}}$):** Angle between the torso vector $\vec{t} = P_{\text{hip}} - P_{\text{shoulder}}$ and upper-arm vector $\vec{b} = P_{\text{elbow}} - P_{\text{shoulder}}$. Intentional shelf reach requires $\theta_{\text{shoulder}} > 38^\circ$, distinguishing it from natural walking swing ($\theta_{\text{shoulder}} < 25^\circ$).

---

### 3. Affective Postural Indices & 3D-Projected Head Orientation
Based on the *Body Action and Posture (BAP) Coding System* (*Dael et al., 2012*; *Kleinsmith & Bianchi-Berthouze, 2013*):

* **Bimanual Convergence Index ($I_{\text{bimanual}}$):** Measures two-handed product manipulation in front of the torso (*Frontoni et al., 2014*):
  $$I_{\text{bimanual}} = \frac{\Vert{} P_{10} - P_9 \Vert{}_2}{W_s} < 0.85$$
* **Cephalic Self-Adaptor & Cognitive Hesitation ($I_{\text{head-adaptor}}$ & $\sigma_{\text{yaw}}$):** Hand-to-head/chin proximity ($I_{\text{head-adaptor}} = \frac{\min \Vert{}P_{\text{wrist}} - P_{\text{head}}\Vert{}_2}{W_s} < 0.72$) or rapid horizontal head oscillation ($\sigma_{\text{yaw}} > 13.5^\circ$ over a sliding window) indicates product comparison, doubt, or cognitive evaluation.
* **Back-Turned 3D Head Yaw Estimation:** When facial keypoints are occluded (shopper facing the shelf), head rotation ($\text{Yaw}$) relative to the torso normal vector is estimated from ear detection asymmetry ($c_{\text{ear\_L}} - c_{\text{ear\_R}}$) and cervical lateral offset relative to the shoulder midpoint.

| Behavioral State | Biomechanical & Kinematic Decision Rule | Primary Literature Reference |
| :--- | :--- | :--- |
| **Interacting / Product Pickup** | ($P_{\text{wrist}} \in \text{ROI}_{\text{prod}}$) $\lor$ ($R_{\text{reach}} > 1.10 \land \theta_{\text{elbow}} > 130^\circ \land \theta_{\text{shoulder}} > 38^\circ$) | *Singh et al. (2016); McAtamney & Corlett (1993)* |
| **Inspecting / Hesitation** | $v_{\text{cm}} < 0.048 \land \left[ (I_{\text{bimanual}} < 0.85 \land \theta_{\text{elbow}} \le 115^\circ) \lor (I_{\text{head-adaptor}} < 0.72) \lor \sigma_{\text{yaw}} > 13.5^\circ \right]$ | *Dael et al. (2012); Frontoni et al. (2014)* |
| **Standing & Analyzing Shelf** | $v_{\text{cm}} < 0.048 \land \text{Gaze} \in \text{ROI}_{\text{prod}} \text{ for } \Delta t \ge 300\text{ms}$ | *Popa et al. (2013); Karg et al. (2013)* |
| **Walking & Browsing Shelf** | $v_{\text{cm}} \ge 0.048 \land \text{Gaze} \in \text{ROI}_{\text{prod}}$ | *Singh et al. (2016)* |

In [11]:
def compute_joint_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """Computes the internal angle (in degrees) at vertex 'b' formed by segments ba and bc."""
    ba, bc = a - b, c - b
    norm_ba, norm_bc = np.linalg.norm(ba), np.linalg.norm(bc)
    if norm_ba < 1e-4 or norm_bc < 1e-4:
        return 180.0
    cos_angle = np.dot(ba, bc) / (norm_ba * norm_bc)
    return float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))


class BiomechanicalKinematicsEngine:
    """
    Computes 3D-projected head gaze cones, scale-invariant biomechanical metrics (RULA/MERL/BAP),
    and opportunistic facial emotion recognition from 17 COCO keypoints.
    """
    def __init__(self, window_size: int = 15, ema_alpha: float = 0.35):
        self.window_size = window_size
        self.ema_alpha = ema_alpha
        self.pos_history: Dict[str, deque] = {}
        self.yaw_history: Dict[str, deque] = {}
        self.gaze_vec_ema: Dict[str, np.ndarray] = {}

    def estimate_head_pose_and_cone(
        self, track_id: str, pts: np.ndarray, confs: np.ndarray, ws: float, shoulder_center: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]:
        """Estimates 3D head orientation (Yaw, Pitch, Roll) and projects a 2D FOV cone."""
        head_pts = [pts[j] for j in range(5) if confs[j] > 0.30 and pts[j][0] > 0]
        head_origin = np.mean(head_pts, axis=0) if head_pts else shoulder_center - np.array([0.0, ws * 0.35])

        if confs[5] > 0.3 and confs[6] > 0.3:
            sh_axis = pts[6] - pts[5]
            dx, dy = sh_axis / (np.linalg.norm(sh_axis) + 1e-5)
            chest_normal = np.array([dy, -dx])
        else:
            chest_normal = np.array([0.0, -1.0])

        face_pts = [pts[j] for j in [0, 1, 2] if confs[j] > 0.32 and pts[j][0] > 0]
        ear_pts = [pts[j] for j in [3, 4] if confs[j] > 0.30 and pts[j][0] > 0]

        if face_pts and ear_pts:
            head_vec = np.mean(face_pts, axis=0) - np.mean(ear_pts, axis=0)
            norm_hv = np.linalg.norm(head_vec)
            raw_dir = (0.85 * (head_vec / norm_hv) + 0.15 * chest_normal) if norm_hv > 2.5 else chest_normal
            yaw_deg = float(np.degrees(np.arctan2(raw_dir[0], -raw_dir[1])))
            pitch_deg = float(np.clip(((head_origin[1] - shoulder_center[1]) / ws + 0.4) * 45.0, -45.0, 45.0))
            roll_deg = float(np.degrees(np.arctan2(pts[2][1] - pts[1][1], pts[2][0] - pts[1][0]))) if (confs[1] > 0.35 and confs[2] > 0.35) else 0.0
            method, quality = "3d_face_sphere", "high"
        else:
            # Back-turned 3D heuristic: combines ear confidence asymmetry and cervical lateral offset
            ear_asym = float(confs[3] - confs[4])
            cervical_offset = float((head_origin[0] - shoulder_center[0]) / ws)
            yaw_rad = np.clip(ear_asym * 0.9 + cervical_offset * 1.4, -1.1, 1.1)
            cos_y, sin_y = np.cos(yaw_rad), np.sin(yaw_rad)
            raw_dir = np.array([
                chest_normal[0] * cos_y - chest_normal[1] * sin_y,
                chest_normal[0] * sin_y + chest_normal[1] * cos_y
            ])
            yaw_deg = float(np.degrees(yaw_rad))
            pitch_deg, roll_deg = -12.0, 0.0
            method, quality = "3d_back_yaw", "medium"

        norm_dir = np.linalg.norm(raw_dir)
        unit_dir = (raw_dir / norm_dir) if norm_dir > 1e-4 else np.array([0.0, -1.0])

        if track_id in self.gaze_vec_ema:
            smooth_dir = self.ema_alpha * unit_dir + (1.0 - self.ema_alpha) * self.gaze_vec_ema[track_id]
            smooth_dir /= (np.linalg.norm(smooth_dir) + 1e-5)
        else:
            smooth_dir = unit_dir
        self.gaze_vec_ema[track_id] = smooth_dir

        gaze_range = ws * 2.5
        gaze_target = head_origin + smooth_dir * gaze_range

        fov_rad = np.radians(18)
        cf, sf = np.cos(fov_rad), np.sin(fov_rad)
        left_ray = np.array([smooth_dir[0]*cf - smooth_dir[1]*sf, smooth_dir[0]*sf + smooth_dir[1]*cf])
        right_ray = np.array([smooth_dir[0]*cf + smooth_dir[1]*sf, -smooth_dir[0]*sf + smooth_dir[1]*cf])
        cone_polygon = np.array([
            head_origin, head_origin + left_ray * gaze_range, gaze_target, head_origin + right_ray * gaze_range
        ], dtype=np.int32)

        head_pose_meta = {
            "yaw_estimated": round(yaw_deg, 1),
            "pitch_estimated": round(pitch_deg, 1),
            "roll_estimated": round(roll_deg, 1),
            "method": method,
            "quality": quality
        }
        return head_origin, gaze_target, cone_polygon, head_pose_meta

    def analyze_facial_expression(self, frame: np.ndarray, pts: np.ndarray, confs: np.ndarray) -> Dict[str, Any]:
        """Runs opportunistic facial emotion inference when nose and eyes are visible."""
        nose_confidence, left_eye_confidence, right_eye_confidence = confs[0:3]
        if nose_confidence < 0.40 or (left_eye_confidence < 0.38 and right_eye_confidence < 0.38):
            return {"visible": False, "expression_detected": None, "raw_emotion": "Occluded (Back)", "confidence": 0.0, "valence": 0.0, "quality": "occluded", "occlusion": 0.85}

        valid_pts = [pt for pt, c in zip(pts[0:5], confs[0:5]) if c >= 0.35 and pt[0] > 0]
        if len(valid_pts) < 2:
            return {"visible": False, "expression_detected": None, "raw_emotion": "Occluded", "confidence": 0.0, "valence": 0.0, "quality": "low", "occlusion": 0.75}

        pts_arr = np.array(valid_pts)
        cx, cy = np.mean(pts_arr, axis=0)
        radius = max(np.max(np.ptp(pts_arr, axis=0)) * 1.5, 24.0)
        h, w = frame.shape[:2]
        x1, y1 = max(0, int(cx - radius)), max(0, int(cy - radius))
        x2, y2 = min(w, int(cx + radius)), min(h, int(cy + radius))

        quality = "good" if (x2 - x1) >= 36 and nose_confidence > 0.60 else "medium"
        occlusion = round(float(1.0 - np.mean([nose_confidence, max(left_eye_confidence, right_eye_confidence)])), 2)

        if HSEMOTION_AVAILABLE and (x2 - x1) >= 18 and (y2 - y1) >= 18:
            try:
                crop_rgb = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
                emo, scores = FACE_RECOGNIZER.predict_emotions(crop_rgb, logits=False)
                conf_score = float(np.max(scores)) if scores is not None else 0.75
                emotion_map = {
                    "Happiness": ("smile", 1.0), "Surprise": ("surprise", 0.8), "Neutral": ("neutral", 0.1),
                    "Sadness": ("frown", -0.4), "Anger": ("frown", -0.8), "Disgust": ("disgust", -0.7),
                    "Fear": ("hesitation", -0.5), "Contempt": ("hesitation", -0.5)
                }
                exp_label, valence = emotion_map.get(emo, ("neutral", 0.1))
                return {
                    "visible": True, "expression_detected": exp_label, "raw_emotion": emo,
                    "confidence": round(conf_score, 2), "valence": valence,
                    "quality": quality, "occlusion": occlusion, "crop_box": (x1, y1, x2, y2)
                }
            except Exception:
                pass

        return {"visible": True, "expression_detected": "neutral", "raw_emotion": "Neutral", "confidence": round(float(nose_confidence), 2), "valence": 0.1, "quality": quality, "occlusion": occlusion, "crop_box": (x1, y1, x2, y2)}

## Production Data Engineering: Temporal Event Processor & ERD Schema

In production environments, raw frame telemetry (`FRAME_OBSERVATION`) is noisy and high-volume. The `RetailVisionPipeline` class below implements a **stateful Temporal Processor** that consolidates frame-level observations into structured relational entities matching our PostgreSQL production schema:

* **`CAMERA` & `SHELF`:** Stores normalized polygon coordinates for both product shelves (`product_roi`) and floor presence zones (`presence_roi`).
* **`FRAME_OBSERVATION`:** Frame-level JSONB telemetry containing normalized bounding boxes, keypoints, 3D head pose, hand ROI collisions, and biomechanical angles.
* **`SHELF_INTERACTION`:** Session entity triggered when a shopper steps into a shelf's presence zone, looks at it, or reaches toward it.
* **`ATTENTION_EPISODE`, `ACTION_EVENT`, & `REACTION_SIGNAL`:** Aggregated temporal episodes with start/end timestamps and duration in milliseconds.
* **`SHELF_METRIC` & `SHELF_SIGNAL_METRIC`:** Pre-aggregated business KPIs ready for analytical dashboards.

In [12]:
class RetailVisionPipeline:
    """
    End-to-end computer vision and temporal analytics pipeline.
    Processes video streams, renders a rich biomechanical HUD, and populates ERD tables.
    """
    def __init__(self, camera_name: str, stream_url: str, shelves_config: List[Dict[str, Any]]):
        self.camera_id = str(uuid.uuid4())
        self.pose_model_id = str(uuid.uuid4())
        self.face_model_id = str(uuid.uuid4())
        self.base_time = datetime.now(timezone.utc)
        self.kinematics = BiomechanicalKinematicsEngine(window_size=15, ema_alpha=0.35)

        self.model_config = {
            "model": {"name": "yolo11s-pose", "version": "11", "input_size": 640, "device": DEVICE},
            "detection": {"person_confidence_threshold": 0.35, "nms_iou_threshold": 0.45},
            "tracking": {"tracker": "bytetrack", "persist": True},
            "processing": {"target_inference_fps": 15, "observation_sample_fps": 15},
            "interaction_rules": {"entry_confirmation_ms": 300, "exit_tolerance_ms": 1500, "tracking_loss_tolerance_ms": 3000}
        }

        self.tb_camera = [{
            "id": self.camera_id, "name": camera_name, "stream_url": stream_url,
            "is_active": True, "created_at": self.base_time.isoformat(), "updated_at": self.base_time.isoformat()
        }]
        self.tb_model_version = [
            {"id": self.pose_model_id, "name": "yolo11s-pose + biomechanics", "kind": "POSE_GAZE_ACTION", "version": "11.1", "configuration": self.model_config, "created_at": self.base_time.isoformat()},
            {"id": self.face_model_id, "name": "hsemotion-enet-b0", "kind": "FACIAL_REACTION", "version": "1.0", "configuration": {"backend": "onnx"}, "created_at": self.base_time.isoformat()}
        ]

        self.tb_shelf: List[Dict] = []
        self.shelves_geom: Dict[str, Dict] = {}
        for g in shelves_config:
            gid = str(uuid.uuid4())
            self.tb_shelf.append({
                "id": gid, "camera_id": self.camera_id, "name": g["name"], "category": g.get("category", "General"),
                "product_roi": g["product_roi"], "presence_roi": g["presence_roi"],
                "roi_version": 1, "is_active": True, "created_at": self.base_time.isoformat(), "updated_at": self.base_time.isoformat()
            })
            self.shelves_geom[gid] = {
                "name": g["name"], "color_bgr": g.get("color_bgr", (0, 165, 255)),
                "pts_prod_norm": g["product_roi"]["points"], "pts_pres_norm": g["presence_roi"]["points"]
            }

        # Relational ERD Tables + Flat Analytics Log
        self.tb_frame_observation: List[Dict] = []
        self.tb_shelf_interaction: List[Dict] = []
        self.tb_observation_interaction: List[Dict] = []
        self.tb_attention_episode: List[Dict] = []
        self.tb_action_event: List[Dict] = []
        self.tb_reaction_signal: List[Dict] = []
        self.tb_shelf_metric: List[Dict] = []
        self.tb_shelf_signal_metric: List[Dict] = []
        self.flat_analytics_records: List[Dict] = []
        self.frame_zero: Optional[np.ndarray] = None

    def _build_pixel_polygons(self, w: int, h: int) -> None:
        for g in self.shelves_geom.values():
            prod_px = [(int(x * w), int(y * h)) for x, y in g["pts_prod_norm"]]
            pres_px = [(int(x * w), int(y * h)) for x, y in g["pts_pres_norm"]]
            g["poly_prod"] = Polygon(prod_px)
            g["poly_pres"] = Polygon(pres_px)
            g["cv2_prod"] = np.array(prod_px, np.int32).reshape((-1, 1, 2))
            g["cv2_pres"] = np.array(pres_px, np.int32).reshape((-1, 1, 2))

    def process_video(self, video_path: str, max_frames: int = 950, output_path: str = "processed_output.mp4") -> List[Tuple]:
        cap = cv2.VideoCapture(video_path)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 15.0
        total_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), max_frames)
        dt_ms = int(1000.0 / fps)

        self._build_pixel_polygons(w, h)
        POSE_MODEL.predictor = None
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

        track_uuids: Dict[str, str] = {}
        focus_ms_tracker: Dict[str, Dict[str, int]] = {}
        active_interactions: Dict[Tuple[str, str], Dict] = {}
        active_attention: Dict[Tuple[str, str], Dict] = {}
        active_actions: Dict[Tuple[str, str], Dict] = {}
        active_reactions: Dict[Tuple[str, str, str], Dict] = {}
        captured_face_crops: List[Tuple] = []

        print(f"Processing '{video_path}' ({total_frames} frames @ {fps:.1f} FPS | {w}x{h}px)...")

        for f_seq in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break
            if f_seq == 0:
                self.frame_zero = frame.copy()

            ts_now = self.base_time + timedelta(milliseconds=f_seq * dt_ms)
            canvas = frame.copy()
            roi_overlay = frame.copy()

            # Draw Planogram ROIs
            for g in self.shelves_geom.values():
                cv2.fillPoly(roi_overlay, [g["cv2_prod"]], g["color_bgr"])
                cv2.polylines(canvas, [g["cv2_prod"]], True, g["color_bgr"], 2, cv2.LINE_AA)
                cv2.polylines(canvas, [g["cv2_pres"]], True, (180, 255, 180), 1, cv2.LINE_AA)
            cv2.addWeighted(roi_overlay, 0.22, canvas, 0.78, 0, canvas)

            for g in self.shelves_geom.values():
                c_pt = g["poly_prod"].centroid
                cx, cy = int(c_pt.x), int(c_pt.y)
                (tw, th), _ = cv2.getTextSize(g["name"], cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
                tx, ty = max(5, min(cx - tw // 2, w - tw - 5)), max(18, min(cy, h - 10))
                cv2.rectangle(canvas, (tx - 4, ty - th - 4), (tx + tw + 4, ty + 4), (20, 20, 20), -1)
                cv2.putText(canvas, g["name"], (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.42, g["color_bgr"], 1, cv2.LINE_AA)

            cone_overlay = canvas.copy()
            res = POSE_MODEL.track(frame, persist=True, device=DEVICE, verbose=False, conf=0.35)[0]

            if res.boxes is not None and res.keypoints is not None and len(res.boxes) > 0:
                boxes = res.boxes.xyxy.cpu().numpy()
                box_confs = res.boxes.conf.cpu().numpy()
                ids = res.boxes.id.cpu().numpy().astype(int) if res.boxes.id is not None else list(range(len(boxes)))
                kpts_xy = res.keypoints.xy.cpu().numpy()
                kpts_conf = res.keypoints.conf.cpu().numpy()

                for i, tid in enumerate(ids):
                    tid_str = f"P{tid:02d}"
                    if tid_str not in track_uuids:
                        track_uuids[tid_str] = str(uuid.uuid4())
                        self.kinematics.pos_history[tid_str] = deque(maxlen=15)
                        self.kinematics.yaw_history[tid_str] = deque(maxlen=15)
                        focus_ms_tracker[tid_str] = {}

                    pts, confs = kpts_xy[i], kpts_conf[i]
                    x1, y1, x2, y2 = boxes[i]

                    # 1. Anatomical Reference Scale (Ws) & Foot Center
                    feet_pts = [pts[idx] for idx in [15, 16] if confs[idx] > 0.30 and pts[idx][0] > 0]
                    foot_px = np.mean(feet_pts, axis=0) if feet_pts else np.array([(x1 + x2) / 2.0, y2])
                    ws = float(np.linalg.norm(pts[6] - pts[5])) if (confs[5] > 0.3 and confs[6] > 0.3 and pts[5][0] > 0 and pts[6][0] > 0) else max((x2 - x1) * 0.60, 28.0)
                    ws = max(ws, 22.0)
                    sh_center = (pts[5] + pts[6]) / 2.0 if (confs[5] > 0.3 and confs[6] > 0.3) else np.array([(x1 + x2) / 2.0, y1 + (y2 - y1) * 0.25])

                    # 2. Locomotion Velocity
                    self.kinematics.pos_history[tid_str].append(foot_px)
                    pos_hist = self.kinematics.pos_history[tid_str]
                    vel_px = float(np.linalg.norm(pos_hist[-1] - pos_hist[0]) / len(pos_hist)) if len(pos_hist) >= 3 else 0.0
                    v_cm_norm = round(vel_px / ws, 4)
                    is_standing = v_cm_norm < 0.048
                    mov_state = "STANDING" if is_standing else "WALKING"

                    # 3. 3D Head Orientation & Gaze Cone
                    head_origin, gaze_target, cone_pts, hp_meta = self.kinematics.estimate_head_pose_and_cone(
                        tid_str, pts, confs, ws, sh_center
                    )
                    self.kinematics.yaw_history[tid_str].append(hp_meta["yaw_estimated"])
                    sigma_yaw = float(np.std(self.kinematics.yaw_history[tid_str])) if len(self.kinematics.yaw_history[tid_str]) >= 5 else 0.0

                    # 4. Spatial Ray-Casting & ROI Collisions
                    gaze_line = LineString([tuple(head_origin), tuple(gaze_target)])
                    looked_gid, reached_gid, presence_gid = None, None, None
                    min_gaze_dist = float("inf")

                    pt_foot = Point(float(foot_px[0]), float(foot_px[1]))
                    pt_head = Point(float(head_origin[0]), float(head_origin[1]))
                    pt_wrist_l = Point(float(pts[9][0]), float(pts[9][1])) if (confs[9] > 0.32 and pts[9][0] > 0) else None
                    pt_wrist_r = Point(float(pts[10][0]), float(pts[10][1])) if (confs[10] > 0.32 and pts[10][0] > 0) else None
                    left_in_roi, right_in_roi = False, False

                    for gid, g in self.shelves_geom.items():
                        if g["poly_pres"].contains(pt_foot) or g["poly_prod"].distance(pt_foot) < ws * 1.1:
                            presence_gid = gid
                        if g["poly_prod"].intersects(gaze_line):
                            d_gaze = g["poly_prod"].distance(pt_head)
                            if d_gaze < min_gaze_dist:
                                min_gaze_dist = d_gaze
                                looked_gid = gid
                        if pt_wrist_l and g["poly_prod"].contains(pt_wrist_l):
                            left_in_roi, reached_gid = True, gid
                        if pt_wrist_r and g["poly_prod"].contains(pt_wrist_r):
                            right_in_roi, reached_gid = True, gid

                    if looked_gid:
                        focus_ms_tracker[tid_str][looked_gid] = focus_ms_tracker[tid_str].get(looked_gid, 0) + dt_ms
                        focus_duration_ms = focus_ms_tracker[tid_str][looked_gid]
                    else:
                        focus_ms_tracker[tid_str] = {}
                        focus_duration_ms = 0

                    # 5. Upper-Limb Biomechanics & Facial Emotion
                    face_info = self.kinematics.analyze_facial_expression(frame, pts, confs)
                    if face_info["visible"] and "crop_box" in face_info and len(captured_face_crops) < 8 and f_seq % 15 == 0:
                        bx1, by1, bx2, by2 = face_info["crop_box"]
                        captured_face_crops.append((f_seq, tid_str, face_info["expression_detected"], face_info["confidence"], frame[by1:by2, bx1:bx2].copy()))

                    theta_elbow_l = compute_joint_angle(pts[5], pts[7], pts[9]) if all(confs[[5, 7, 9]] > 0.3) else 165.0
                    theta_elbow_r = compute_joint_angle(pts[6], pts[8], pts[10]) if all(confs[[6, 8, 10]] > 0.3) else 165.0
                    theta_elbow_min = min(theta_elbow_l, theta_elbow_r)
                    theta_elbow_max = max(theta_elbow_l, theta_elbow_r)

                    theta_sh_max = max(
                        compute_joint_angle(pts[11], pts[5], pts[7]) if all(confs[[11, 5, 7]] > 0.3) else 15.0,
                        compute_joint_angle(pts[12], pts[6], pts[8]) if all(confs[[12, 6, 8]] > 0.3) else 15.0
                    )
                    r_reach_max = max(
                        (np.linalg.norm(pts[9] - pts[5]) / ws) if (confs[9] > 0.3 and confs[5] > 0.3) else 0.4,
                        (np.linalg.norm(pts[10] - pts[6]) / ws) if (confs[10] > 0.3 and confs[6] > 0.3) else 0.4
                    )
                    i_bimanual = float(np.linalg.norm(pts[10] - pts[9]) / ws) if (confs[9] > 0.3 and confs[10] > 0.3 and pts[9][0] > 0 and pts[10][0] > 0) else 1.5
                    head_dists = [np.linalg.norm(pts[idx_p] - head_origin) / ws for idx_p in [9, 10] if confs[idx_p] > 0.3 and pts[idx_p][0] > 0]
                    i_head_adaptor = float(min(head_dists)) if head_dists else 2.0

                    # Biomechanical Action Classification
                    is_reaching = (r_reach_max > 1.10 and theta_elbow_max > 130.0 and theta_sh_max > 38.0)
                    is_inspecting = (is_standing and i_bimanual < 0.85 and 50.0 <= theta_elbow_min <= 115.0)
                    is_hesitating = is_standing and ((i_head_adaptor < 0.72 and theta_elbow_min < 108.0) or (sigma_yaw > 13.5 and looked_gid is not None))

                    if reached_gid is not None or (is_reaching and looked_gid is not None):
                        body_action, action_weight = "INTERACTING / PRODUCT PICKUP", 52
                    elif is_hesitating or is_inspecting:
                        body_action, action_weight = "INSPECTING / HESITATION", 42
                    elif is_standing and looked_gid is not None and focus_duration_ms >= 300:
                        body_action, action_weight = "STANDING & ANALYZING SHELF", 34
                    elif looked_gid is not None:
                        body_action, action_weight = "WALKING & BROWSING SHELF", 22
                    elif is_standing:
                        body_action, action_weight = "STANDING IN AISLE", 12
                    else:
                        body_action, action_weight = "PASSING BY", 5

                    focus_bonus = min(30, int((focus_duration_ms / 1000.0) * 12))
                    posture_bonus = 15 if (is_standing and theta_sh_max > 25.0) else (10 if is_standing else 0)
                    engagement_score = int(np.clip(action_weight + focus_bonus + posture_bonus + int(face_info["valence"] * 10), 0, 100))

                    # 6. Populate FRAME_OBSERVATION
                    obs_id = str(uuid.uuid4())
                    obs_attributes = {
                        "bbox": {"x1": round(float(x1/w), 4), "y1": round(float(y1/h), 4), "x2": round(float(x2/w), 4), "y2": round(float(y2/h), 4), "confidence": round(float(box_confs[i]), 2)},
                        "pose": {
                            "nose": [round(float(pts[0][0]/w), 4), round(float(pts[0][1]/h), 4)],
                            "left_shoulder": [round(float(pts[5][0]/w), 4), round(float(pts[5][1]/h), 4)],
                            "right_shoulder": [round(float(pts[6][0]/w), 4), round(float(pts[6][1]/h), 4)],
                            "left_wrist": [round(float(pts[9][0]/w), 4), round(float(pts[9][1]/h), 4)],
                            "right_wrist": [round(float(pts[10][0]/w), 4), round(float(pts[10][1]/h), 4)],
                            "left_ankle": [round(float(pts[15][0]/w), 4), round(float(pts[15][1]/h), 4)],
                            "right_ankle": [round(float(pts[16][0]/w), 4), round(float(pts[16][1]/h), 4)]
                        },
                        "head_pose": hp_meta,
                        "attention": {"direction_detected": True, "focus_duration_ms": int(focus_duration_ms), "shelf_confidence": 0.88 if looked_gid else 0.0},
                        "hands": {
                            "left": {"x": round(float(pts[9][0]/w), 4), "y": round(float(pts[9][1]/h), 4), "in_product_roi": left_in_roi},
                            "right": {"x": round(float(pts[10][0]/w), 4), "y": round(float(pts[10][1]/h), 4), "in_product_roi": right_in_roi}
                        },
                        "movement": {"state": mov_state, "velocity_estimated": v_cm_norm},
                        "face": {"visible": face_info["visible"], "expression_detected": face_info["expression_detected"], "confidence": face_info["confidence"]}
                    }
                    self.tb_frame_observation.append({
                        "id": obs_id, "camera_id": self.camera_id, "attention_shelf_id": looked_gid, "reach_shelf_id": reached_gid,
                        "tracking_session_id": track_uuids[tid_str], "local_track_id": tid_str, "frame_seq": f_seq,
                        "observed_at": ts_now.isoformat(), "attributes": obs_attributes, "created_at": ts_now.isoformat()
                    })

                    # Store flat record for heatmaps and biomechanical plots
                    active_wrist = pts[10] if (confs[10] > 0.32 and pts[10][0] > 0) else (pts[9] if (confs[9] > 0.32 and pts[9][0] > 0) else None)
                    self.flat_analytics_records.append({
                        "frame": f_seq, "timestamp_s": round(f_seq / fps, 2), "person_id": tid_str,
                        "foot_x": int(foot_px[0]), "foot_y": int(foot_px[1]),
                        "gaze_x": int(gaze_target[0]), "gaze_y": int(gaze_target[1]),
                        "hand_x": int(active_wrist[0]) if active_wrist is not None else None,
                        "hand_y": int(active_wrist[1]) if active_wrist is not None else None,
                        "shelf_looked": self.shelves_geom[looked_gid]["name"] if looked_gid else "None",
                        "shelf_reached": self.shelves_geom[reached_gid]["name"] if reached_gid else "None",
                        "body_action": body_action, "facial_emotion": face_info["raw_emotion"],
                        "v_cm_norm_ws": v_cm_norm, "theta_elbow_min": round(theta_elbow_min, 1),
                        "theta_shoulder_max": round(theta_sh_max, 1), "r_reach_norm": round(float(r_reach_max), 2),
                        "i_bimanual": round(i_bimanual, 2), "i_head_adaptor": round(i_head_adaptor, 2),
                        "sigma_yaw_deg": round(sigma_yaw, 2), "engagement_score": engagement_score
                    })

                    # 7. Temporal Event Processor (ERD Entities)
                    ctx_gid = reached_gid or looked_gid or presence_gid
                    if ctx_gid:
                        int_key = (tid_str, ctx_gid)
                        if int_key not in active_interactions:
                            active_interactions[int_key] = {
                                "id": str(uuid.uuid4()), "shelf_id": ctx_gid, "started_at": ts_now.isoformat(), "ended_at": ts_now.isoformat(),
                                "last_presence_at": ts_now.isoformat(), "presence_duration_ms": dt_ms, "status": "COMPLETED",
                                "end_reason": "LEFT_ROI", "roi_version": 1, "created_at": ts_now.isoformat(), "updated_at": ts_now.isoformat()
                            }
                            self.tb_shelf_interaction.append(active_interactions[int_key])
                        else:
                            active_interactions[int_key]["ended_at"] = ts_now.isoformat()
                            active_interactions[int_key]["last_presence_at"] = ts_now.isoformat()
                            active_interactions[int_key]["presence_duration_ms"] += dt_ms

                        int_id = active_interactions[int_key]["id"]
                        self.tb_observation_interaction.append({
                            "observation_id": obs_id, "interaction_id": int_id,
                            "link_type": "REACH" if reached_gid else ("ATTENTION" if looked_gid else "PRESENCE"),
                            "created_at": ts_now.isoformat()
                        })

                        if looked_gid == ctx_gid and focus_duration_ms >= 200:
                            if int_key not in active_attention:
                                ep_obj = {"id": str(uuid.uuid4()), "interaction_id": int_id, "model_id": self.pose_model_id, "started_at": ts_now.isoformat(), "ended_at": ts_now.isoformat(), "method": hp_meta["method"], "confidence": 0.86, "created_at": ts_now.isoformat()}
                                active_attention[int_key] = ep_obj
                                self.tb_attention_episode.append(ep_obj)
                            else:
                                active_attention[int_key]["ended_at"] = ts_now.isoformat()

                        if reached_gid or is_reaching or is_inspecting:
                            hand_side = "right" if (right_in_roi or confs[10] >= confs[9]) else "left"
                            act_type = "PICKUP" if (left_in_roi or right_in_roi) else ("INSPECT" if is_inspecting else "REACH")
                            if int_key not in active_actions:
                                act_obj = {
                                    "id": str(uuid.uuid4()), "interaction_id": int_id, "model_id": self.pose_model_id,
                                    "action_type": act_type, "occurred_at": ts_now.isoformat(), "confidence": 0.89,
                                    "attributes": {
                                        "hand": hand_side, "detection_method": "pose_object_tracking",
                                        "object": {"detected": True, "class": "product", "product_id": None, "sku": None, "tracking_confidence": 0.89},
                                        "interaction": {"hand_entered_roi": bool(left_in_roi or right_in_roi), "object_displacement_detected": True, "object_left_shelf": bool(left_in_roi or right_in_roi)},
                                        "temporal_evidence": {"start_frame": f_seq, "end_frame": f_seq, "duration_ms": dt_ms}
                                    }, "created_at": ts_now.isoformat()
                                }
                                active_actions[int_key] = act_obj
                                self.tb_action_event.append(act_obj)
                            else:
                                active_actions[int_key]["attributes"]["temporal_evidence"]["end_frame"] = f_seq
                                active_actions[int_key]["attributes"]["temporal_evidence"]["duration_ms"] += dt_ms

                        if face_info["visible"] or is_standing:
                            sig_type = face_info["expression_detected"].upper() if face_info["visible"] else body_action
                            sig_key = (tid_str, ctx_gid, sig_type)
                            if sig_key not in active_reactions:
                                sig_obj = {
                                    "id": str(uuid.uuid4()), "interaction_id": int_id, "model_id": self.face_model_id,
                                    "signal_type": sig_type, "source": "FACE_AND_BODY" if face_info["visible"] else "BODY_BIOMECHANICS",
                                    "occurred_at": ts_now.isoformat(), "confidence": face_info["confidence"] if face_info["visible"] else 0.82,
                                    "attributes": {
                                        "facial_analysis": {"face_visible": face_info["visible"], "detected_expression": face_info["expression_detected"], "visibility_quality": face_info["quality"], "occlusion_estimated": face_info["occlusion"]},
                                        "behavioral_context": {"person_standing": is_standing, "attention_detected": bool(looked_gid), "handling_product": bool(reached_gid or is_inspecting)},
                                        "temporal_evidence": {"start_frame": f_seq, "end_frame": f_seq, "duration_ms": dt_ms},
                                        "association": {"single_shelf_context": True, "context_quality": "high" if looked_gid else "medium"},
                                        "inferred_emotion": face_info["raw_emotion"]
                                    }, "created_at": ts_now.isoformat()
                                }
                                active_reactions[sig_key] = sig_obj
                                self.tb_reaction_signal.append(sig_obj)
                            else:
                                active_reactions[sig_key]["attributes"]["temporal_evidence"]["end_frame"] = f_seq
                                active_reactions[sig_key]["attributes"]["temporal_evidence"]["duration_ms"] += dt_ms

                    # 8. Render HUD (3D Gaze Cone + Skeleton + Engagement Progress Bar)
                    gaze_color = (0, 255, 0) if looked_gid else (0, 215, 255)
                    cv2.fillPoly(cone_overlay, [cone_pts], gaze_color)
                    cv2.arrowedLine(canvas, (int(head_origin[0]), int(head_origin[1])), (int(gaze_target[0]), int(gaze_target[1])), gaze_color, 2, cv2.LINE_AA, tipLength=0.14)

                    for p1, p2 in [(5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12)]:
                        if confs[p1] > 0.30 and confs[p2] > 0.30:
                            cv2.line(canvas, (int(pts[p1][0]), int(pts[p1][1])), (int(pts[p2][0]), int(pts[p2][1])), (0, 255, 180), 2, cv2.LINE_AA)

                    if face_info["visible"] and "crop_box" in face_info:
                        fx1, fy1, fx2, fy2 = face_info["crop_box"]
                        cv2.rectangle(canvas, (fx1, fy1), (fx2, fy2), (255, 0, 255), 2)

                    bar_color = (0, 255, 0) if engagement_score >= 55 else ((0, 200, 255) if engagement_score >= 30 else (180, 180, 180))
                    yb = max(46, int(y1))
                    xb = max(5, min(int(x1), w - 305))
                    cv2.rectangle(canvas, (xb, yb - 42), (xb + 300, yb), (15, 15, 15), -1)
                    cv2.rectangle(canvas, (xb, yb - 5), (xb + int(3.0 * engagement_score), yb), bar_color, -1)
                    cv2.putText(canvas, f"{tid_str} | {body_action}", (xb + 5, yb - 24), cv2.FONT_HERSHEY_SIMPLEX, 0.42, bar_color, 1, cv2.LINE_AA)
                    cv2.putText(canvas, f"Engage: {engagement_score}% | Face: {face_info['raw_emotion']}", (xb + 5, yb - 9), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 255, 255), 1, cv2.LINE_AA)

            cv2.addWeighted(cone_overlay, 0.26, canvas, 0.74, 0, canvas)
            writer.write(canvas)

        cap.release()
        writer.release()
        self._aggregate_shelf_metrics(total_frames * dt_ms)
        print(f"Done! Frame Observations: {len(self.tb_frame_observation)} | Shelf Interactions: {len(self.tb_shelf_interaction)} | Reaction Signals: {len(self.tb_reaction_signal)}")
        return captured_face_crops

    def _aggregate_shelf_metrics(self, total_duration_ms: int) -> None:
        window_end = (self.base_time + timedelta(milliseconds=total_duration_ms)).isoformat()
        for g in self.tb_shelf:
            gid = g["id"]
            g_ints = [x for x in self.tb_shelf_interaction if x["shelf_id"] == gid]
            int_ids = {x["id"] for x in g_ints}
            g_eps = [x for x in self.tb_attention_episode if x["interaction_id"] in int_ids]
            g_acts = [x for x in self.tb_action_event if x["interaction_id"] in int_ids]
            g_sigs = [x for x in self.tb_reaction_signal if x["interaction_id"] in int_ids]

            pres_ms = sum(x["presence_duration_ms"] for x in g_ints)
            att_ms = sum(int((datetime.fromisoformat(e["ended_at"]) - datetime.fromisoformat(e["started_at"])).total_seconds() * 1000) + 200 for e in g_eps)
            self.tb_shelf_metric.append({
                "shelf_id": gid, "shelf_name": g["name"], "granularity": "SESSION_WINDOW",
                "window_start": self.base_time.isoformat(), "window_end": window_end,
                "interactions_started": len(g_ints), "interactions_with_attention": len({e["interaction_id"] for e in g_eps}),
                "attention_episodes": len(g_eps), "presence_duration_ms": pres_ms, "attention_duration_ms": att_ms,
                "reach_count": sum(1 for a in g_acts if a["action_type"] in ["REACH", "INSPECT"]),
                "touch_count": sum(1 for a in g_acts if a["action_type"] == "PICKUP"),
                "pickup_count": sum(1 for a in g_acts if a["action_type"] == "PICKUP"),
                "return_count": 0, "updated_at": window_end
            })
            sig_counts: Dict[str, int] = {}
            for s in g_sigs:
                sig_counts[s["signal_type"]] = sig_counts.get(s["signal_type"], 0) + 1
            for s_type, cnt in sig_counts.items():
                self.tb_shelf_signal_metric.append({
                    "shelf_id": gid, "shelf_name": g["name"], "granularity": "SESSION_WINDOW",
                    "window_start": self.base_time.isoformat(), "signal_type": s_type, "signal_count": cnt, "updated_at": window_end
                })

## Demonstration 1: Retail Store Overview

Process a bounded excerpt of the source video linked below, with five configured
product and presence zones. The processed export contains 950 annotated frames
(31.7 seconds at 30 FPS). Results are exploratory and use the notebook's heuristic
rules; they are not an accuracy benchmark.


In [ ]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=-1bRhYjw1qE"
VIDEO_PATH = "acti_retail_store_usa.mp4"

if not os.path.exists(VIDEO_PATH):
    subprocess.run([
        "yt-dlp", "-f", "bestvideo[ext=mp4][height<=720]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "--merge-output-format", "mp4", "-o", VIDEO_PATH, YOUTUBE_URL
    ], check=True)

SHELVES_CONFIG = [
    {
        "name": "Center-Left Aisle (Snacks & Candy)",
        "category": "Confectionery",
        "color_bgr": (0, 165, 255),
        "product_roi": {"type": "polygon", "coordinate_system": "normalized", "points": [[0.28, 0.25], [0.58, 0.14], [0.58, 0.42], [0.32, 0.77]]},
        "presence_roi": {"type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center", "points": [[0.32, 0.77], [0.58, 0.42], [0.64, 0.52], [0.42, 0.88]]}
    },
    {
        "name": "Front Endcap Display (Sprays/Promo)",
        "category": "Promotional Endcap",
        "color_bgr": (147, 20, 255),
        "product_roi": {"type": "polygon", "coordinate_system": "normalized", "points": [[0.45, 0.56], [0.57, 0.56], [0.57, 0.96], [0.45, 0.96]]},
        "presence_roi": {"type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center", "points": [[0.36, 0.75], [0.65, 0.75], [0.65, 0.99], [0.36, 0.99]]}
    },
    {
        "name": "Center-Right Aisle (Grocery)",
        "category": "Grocery",
        "color_bgr": (255, 191, 0),
        "product_roi": {"type": "polygon", "coordinate_system": "normalized", "points": [[0.54, 0.30], [0.78, 0.22], [0.79, 0.65], [0.56, 0.95]]},
        "presence_roi": {"type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center", "points": [[0.42, 0.85], [0.62, 0.45], [0.85, 0.68], [0.62, 0.98]]}
    },
    {
        "name": "Back Wall Coolers (Beverages)",
        "category": "Cold Beverages",
        "color_bgr": (50, 205, 50),
        "product_roi": {"type": "polygon", "coordinate_system": "normalized", "points": [[0.38, 0.01], [0.99, 0.01], [0.99, 0.52], [0.38, 0.22]]},
        "presence_roi": {"type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center", "points": [[0.38, 0.20], [0.99, 0.45], [0.99, 0.62], [0.38, 0.32]]}
    },
    {
        "name": "Left Checkout & Coffee Counter",
        "category": "Point of Sale",
        "color_bgr": (0, 215, 255),
        "product_roi": {"type": "polygon", "coordinate_system": "normalized", "points": [[0.01, 0.15], [0.22, 0.15], [0.22, 0.72], [0.01, 0.72]]},
        "presence_roi": {"type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center", "points": [[0.02, 0.45], [0.30, 0.45], [0.30, 0.85], [0.02, 0.85]]}
    }
]

pipeline = RetailVisionPipeline(
    camera_name="ACTI_E43A_RETAIL_USA_02",
    stream_url=YOUTUBE_URL,
    shelves_config=SHELVES_CONFIG
)

face_crops = pipeline.process_video(
    video_path=VIDEO_PATH,
    max_frames=950,
    output_path="retailpose_raw.mp4"
)

if face_crops:
    fig, axes = plt.subplots(1, len(face_crops), figsize=(3.0 * len(face_crops), 3.2))
    if len(face_crops) == 1:
        axes = [axes]
    for idx, (f_id, tid, expr, conf, crop_img) in enumerate(face_crops):
        axes[idx].imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f"{tid} (Frame {f_id})\n{expr} ({int(conf*100)}%)", fontsize=9, fontweight="bold")
        axes[idx].axis("off")
    plt.suptitle("Opportunistic Facial Expression Crops (HSEmotion)", fontsize=13, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()

subprocess.run(["ffmpeg", "-y", "-i", "retailpose_raw.mp4", "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-loglevel", "error", "retailpose_h264.mp4"], check=True)
Video("retailpose_h264.mp4", embed=False, width=900)


## Relational Database Outputs & JSONB Telemetry Verification

Below we inspect the consolidated database records generated by the Temporal Processor (`SHELF_METRIC`, `SHELF_SIGNAL_METRIC`, and `SHELF_INTERACTION`) alongside sample JSONB payloads ready for PostgreSQL ingestion.

In [14]:
print("1. SHELF_METRIC (Consolidated Shelf Performance):")
display(pd.DataFrame(pipeline.tb_shelf_metric)[[
    "shelf_name", "interactions_started", "interactions_with_attention",
    "attention_episodes", "presence_duration_ms", "attention_duration_ms", "reach_count", "touch_count"
]])

print("\n2. SHELF_SIGNAL_METRIC (Facial & Postural Signals per Shelf):")
display(pd.DataFrame(pipeline.tb_shelf_signal_metric)[["shelf_name", "signal_type", "signal_count"]])

print("\n3. Sample Payload -> frame_observation.attributes:")
sample_obs = next((x for x in pipeline.tb_frame_observation if x["attributes"]["face"]["visible"]), pipeline.tb_frame_observation[0])
print(json.dumps(to_json_native(sample_obs["attributes"]), indent=2, allow_nan=False))

if pipeline.tb_action_event:
    print("\n4. Sample Payload -> action_event.attributes:")
    print(json.dumps(pipeline.tb_action_event[0]["attributes"], indent=2))

1. SHELF_METRIC (Consolidated Shelf Performance):


,shelf_name,interactions_started,interactions_with_attention,attention_episodes,presence_duration_ms,attention_duration_ms,reach_count,touch_count
0,Center-Left Aisle (Snacks & Candy),8,5,5,17094,25057,1,7
1,Front Endcap Display (Sprays/Promo),2,2,2,2475,5548,0,2
2,Center-Right Aisle (Grocery),4,1,1,5313,3137,1,3
3,Back Wall Coolers (Beverages),8,6,6,28248,30438,0,8
4,Left Checkout & Coffee Counter,10,10,10,65604,71828,1,9



2. SHELF_SIGNAL_METRIC (Facial & Postural Signals per Shelf):


,shelf_name,signal_type,signal_count
0,Center-Left Aisle (Snacks & Candy),INTERACTING / PRODUCT PICKUP,4
1,Center-Left Aisle (Snacks & Candy),NEUTRAL,5
2,Center-Left Aisle (Snacks & Candy),SMILE,3
3,Center-Left Aisle (Snacks & Candy),HESITATION,5
4,Center-Left Aisle (Snacks & Candy),SURPRISE,3
5,Center-Left Aisle (Snacks & Candy),FROWN,2
6,Center-Left Aisle (Snacks & Candy),DISGUST,1
7,Center-Left Aisle (Snacks & Candy),INSPECTING / HESITATION,1
8,Center-Left Aisle (Snacks & Candy),STANDING & ANALYZING SHELF,1
9,Front Endcap Display (Sprays/Promo),HESITATION,2



3. Sample Payload -> frame_observation.attributes:
{
  "bbox": {
    "x1": 0.5621,
    "y1": 0.147,
    "x2": 0.6312,
    "y2": 0.3964,
    "confidence": 0.87
  },
  "pose": {
    "nose": [
      0.5843,
      0.1798
    ],
    "left_shoulder": [
      0.6126,
      0.219
    ],
    "right_shoulder": [
      0.579,
      0.2183
    ],
    "left_wrist": [
      0.623,
      0.3279
    ],
    "right_wrist": [
      0.5683,
      0.3247
    ],
    "left_ankle": [
      0.6073,
      0.4513
    ],
    "right_ankle": [
      0.5813,
      0.4485
    ]
  },
  "head_pose": {
    "yaw_estimated": -96.9,
    "pitch_estimated": -14.4,
    "roll_estimated": 179.4,
    "method": "3d_face_sphere",
    "quality": "high"
  },
  "attention": {
    "direction_detected": true,
    "focus_duration_ms": 33,
    "shelf_confidence": 0.88
  },
  "hands": {
    "left": {
      "x": 0.623,
      "y": 0.3279,
      "in_product_roi": true
    },
    "right": {
      "x": 0.5683,
      "y": 0.3247,
      "in_pro

## Spatio-Temporal Heatmaps on `Frame(0)` & Biomechanical Telemetry

Finally, we project Gaussian kernel density heatmaps onto the clean `Frame(0)` reference image—segregated by **Biomechanical Action**—and plot the continuous joint angle trajectories ($\theta_{\text{elbow}}, \theta_{\text{shoulder}}, I_{\text{bimanual}}, v_{\text{cm}}$) across time.

In [15]:
df_analytics = pd.DataFrame(pipeline.flat_analytics_records)

def render_gaussian_heatmap(
    base_frame: np.ndarray, df_sub: pd.DataFrame, col_x: str, col_y: str,
    shelves_geom: Dict, weight_col: Optional[str] = None, sigma: int = 36, max_alpha: float = 0.78
) -> np.ndarray:
    h, w = base_frame.shape[:2]
    acc = np.zeros((h, w), dtype=np.float32)
    valid = df_sub.dropna(subset=[col_x, col_y])

    for _, row in valid.iterrows():
        px, py = int(row[col_x]), int(row[col_y])
        if 0 <= px < w and 0 <= py < h:
            w_val = float(row[weight_col]) if (weight_col and pd.notna(row[weight_col])) else 1.0
            cv2.circle(acc, (px, py), radius=12, color=w_val, thickness=-1)

    if np.max(acc) <= 0:
        return base_frame.copy()

    ksize = int(sigma * 4) | 1
    blurred = cv2.GaussianBlur(acc, (ksize, ksize), sigmaX=sigma, sigmaY=sigma)
    norm_heat = blurred / (np.max(blurred) + 1e-6)
    color_map = cv2.applyColorMap(np.uint8(255 * norm_heat), cv2.COLORMAP_TURBO)

    alpha_mask = np.clip(norm_heat * 1.4, 0.0, 1.0)[..., np.newaxis] * max_alpha
    alpha_mask[norm_heat < 0.05] = 0.0

    blended = (base_frame.astype(np.float32) * (1.0 - alpha_mask) + color_map.astype(np.float32) * alpha_mask).astype(np.uint8)
    for g in shelves_geom.values():
        cv2.polylines(blended, [g["cv2_prod"]], True, (255, 255, 255), 2, cv2.LINE_AA)
    return blended

# 1. Action-Segregated Heatmaps on Frame(0)
actions = df_analytics["body_action"].value_counts().index.tolist()
fig, axes = plt.subplots(1, len(actions), figsize=(5.5 * len(actions), 5.2))
if len(actions) == 1:
    axes = [axes]

for idx, act in enumerate(actions):
    df_act = df_analytics[df_analytics["body_action"] == act]
    hm_img = render_gaussian_heatmap(
        pipeline.frame_zero, df_act, col_x="gaze_x", col_y="gaze_y",
        shelves_geom=pipeline.shelves_geom, weight_col="engagement_score"
    )
    axes[idx].imshow(cv2.cvtColor(hm_img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f"{act}\n({len(df_act)} frames)", fontsize=10, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Frame(0) Spatial Attention Heatmaps Segregated by Biomechanical Action", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# 2. Biomechanical Kinematic Time-Series for Primary Tracked Shopper
top_shopper = df_analytics["person_id"].value_counts().index[0]
df_shopper = df_analytics[df_analytics["person_id"] == top_shopper].sort_values("timestamp_s")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df_shopper["timestamp_s"], df_shopper["theta_elbow_min"], label="Min Elbow Angle (deg)", color="#2980b9", lw=2)
axes[0].plot(df_shopper["timestamp_s"], df_shopper["theta_shoulder_max"], label="Max Shoulder Elevation (deg)", color="#e67e22", lw=2)
axes[0].axhline(115, color="#2980b9", linestyle="--", alpha=0.5, label="Inspection Threshold (<115 deg)")
axes[0].axhline(38, color="#e67e22", linestyle="--", alpha=0.5, label="Reach Threshold (>38 deg)")
axes[0].set_ylabel("Joint Angle (deg)")
axes[0].set_title(f"Biomechanical Telemetry Time-Series — Shopper {top_shopper}", fontsize=12, fontweight="bold")
axes[0].legend(loc="upper right", ncol=2)
axes[0].grid(True, linestyle=":", alpha=0.6)

axes[1].plot(df_shopper["timestamp_s"], df_shopper["r_reach_norm"], label="Reach Ratio (R_reach / Ws)", color="#27ae60", lw=2)
axes[1].plot(df_shopper["timestamp_s"], df_shopper["i_bimanual"], label="Bimanual Convergence (I_bimanual / Ws)", color="#8e44ad", lw=2)
axes[1].plot(df_shopper["timestamp_s"], df_shopper["engagement_score"] / 50.0, label="Engagement Score (Scaled 0-2)", color="#c0392b", lw=1.8, linestyle="-.")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Dimensionless Ratio (Ws units)")
axes[1].legend(loc="upper right", ncol=3)
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### Published POC results: Retail store overview

These figures preserve the completed demonstration run. Rerunning the preceding cell recomputes the plots from the current in-memory records.

![Action-grouped spatial heatmaps](../docs/media/retail-store-heatmaps.png)

![Biomechanical telemetry time series](../docs/media/retail-store-kinematics.png)

Heatmap intensities are normalized separately in each panel, and counts represent track-frame observations. These heuristic outputs have not been validated against ground truth.


## Demonstration 2: Aisle Close-Up

The second source video shows a closer view of a beverage chiller and magazine
racks. It provides another camera perspective for inspecting pose tracks, ROI
alignment, projected head direction and upper-limb indicators. The available
processed export spans 212.5 seconds. Behavioral and expression labels remain
experimental; the clip does not validate a detector or establish product removal.


In [ ]:
YOUTUBE_URL_2 = "https://www.youtube.com/watch?v=B9xQM8GEiYk"
VIDEO_PATH_2 = "iprox_retail_aisle_hd.mp4"

if not os.path.exists(VIDEO_PATH_2):
    subprocess.run([
        "yt-dlp", "-f", "bestvideo[ext=mp4][height<=720]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "--merge-output-format", "mp4", "-o", VIDEO_PATH_2, YOUTUBE_URL_2
    ], check=True)

SHELVES_CONFIG_IPROX = [
    {
        "name": "Left Chiller (Juices & Soft Drinks)",
        "category": "Cold Beverages",
        "color_bgr": (50, 205, 50),
        "product_roi": {
            "type": "polygon", "coordinate_system": "normalized",
            "points": [[0.01, 0.05], [0.39, 0.02], [0.39, 0.68], [0.01, 0.85]]
        },
        "presence_roi": {
            "type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center",
            "points": [[0.05, 0.65], [0.48, 0.55], [0.52, 0.98], [0.05, 0.98]]
        }
    },
    {
        "name": "Floor Multipack Stack (Bottled Water)",
        "category": "Bulk Beverages",
        "color_bgr": (255, 191, 0),
        "product_roi": {
            "type": "polygon", "coordinate_system": "normalized",
            "points": [[0.18, 0.62], [0.37, 0.56], [0.38, 0.82], [0.19, 0.88]]
        },
        "presence_roi": {
            "type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center",
            "points": [[0.15, 0.70], [0.48, 0.62], [0.50, 0.98], [0.15, 0.98]]
        }
    },
    {
        "name": "Right Rack - Upper & Mid Tiers (Magazines)",
        "category": "Print Media & Periodicals",
        "color_bgr": (0, 165, 255),
        "product_roi": {
            "type": "polygon", "coordinate_system": "normalized",
            "points": [[0.39, 0.01], [0.92, 0.01], [0.89, 0.86], [0.39, 0.50]]
        },
        "presence_roi": {
            "type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center",
            "points": [[0.22, 0.55], [0.40, 0.50], [0.75, 0.98], [0.22, 0.98]]
        }
    },
    {
        "name": "Right Rack - Bottom Tier (Newspapers)",
        "category": "Daily Press",
        "color_bgr": (147, 20, 255),
        "product_roi": {
            "type": "polygon", "coordinate_system": "normalized",
            "points": [[0.37, 0.52], [0.85, 0.88], [0.72, 0.99], [0.35, 0.65]]
        },
        "presence_roi": {
            "type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center",
            "points": [[0.20, 0.60], [0.42, 0.58], [0.68, 0.99], [0.20, 0.99]]
        }
    },
    {
        "name": "Foreground Right Endcap (Confectionery)",
        "category": "Impulse & Candy",
        "color_bgr": (0, 215, 255),
        "product_roi": {
            "type": "polygon", "coordinate_system": "normalized",
            "points": [[0.88, 0.35], [0.99, 0.35], [0.99, 0.99], [0.86, 0.99]]
        },
        "presence_roi": {
            "type": "polygon", "coordinate_system": "normalized", "reference_point": "feet_center",
            "points": [[0.55, 0.75], [0.95, 0.75], [0.95, 0.99], [0.55, 0.99]]
        }
    }
]

pipeline_iprox = RetailVisionPipeline(
    camera_name="IPROX_HD_RETAIL_AISLE_01",
    stream_url=YOUTUBE_URL_2,
    shelves_config=SHELVES_CONFIG_IPROX
)

# Process the configured frame budget
face_crops_2 = pipeline_iprox.process_video(
    video_path=VIDEO_PATH_2,
    max_frames=7000,
    output_path="iprox_processed_raw.mp4"
)

if face_crops_2:
    fig, axes = plt.subplots(1, len(face_crops_2), figsize=(3.0 * len(face_crops_2), 3.2))
    if len(face_crops_2) == 1:
        axes = [axes]
    for idx, (f_id, tid, expr, conf, crop_img) in enumerate(face_crops_2):
        axes[idx].imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f"{tid} (Frame {f_id})\n{expr} ({int(conf*100)}%)", fontsize=9, fontweight="bold")
        axes[idx].axis("off")
    plt.suptitle("Opportunistic Facial Expression Crops — Scenario 2 (HSEmotion)", fontsize=13, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()

subprocess.run(["ffmpeg", "-y", "-i", "iprox_processed_raw.mp4", "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-loglevel", "error", "iprox_processed_h264.mp4"], check=True)
Video("iprox_processed_h264.mp4", embed=False, width=900)


In [17]:
print("1. SHELF_METRIC — Scenario 2 (Consolidated Shelf Performance):")
display(pd.DataFrame(pipeline_iprox.tb_shelf_metric)[[
    "shelf_name", "interactions_started", "interactions_with_attention",
    "attention_episodes", "presence_duration_ms", "attention_duration_ms", "reach_count", "touch_count"
]])

print("\n2. SHELF_SIGNAL_METRIC — Scenario 2 (Facial & Postural Signals per Shelf):")
display(pd.DataFrame(pipeline_iprox.tb_shelf_signal_metric)[["shelf_name", "signal_type", "signal_count"]])

print("\n3. Sample Payload (Scenario 2) -> frame_observation.attributes:")
sample_obs_2 = next((x for x in pipeline_iprox.tb_frame_observation if x["attributes"]["face"]["visible"]), pipeline_iprox.tb_frame_observation[0])
print(json.dumps(to_json_native(sample_obs_2["attributes"]), indent=2, allow_nan=False))

1. SHELF_METRIC — Scenario 2 (Consolidated Shelf Performance):


,shelf_name,interactions_started,interactions_with_attention,attention_episodes,presence_duration_ms,attention_duration_ms,reach_count,touch_count
0,Left Chiller (Juices & Soft Drinks),8,7,7,22407,54068,2,4
1,Floor Multipack Stack (Bottled Water),8,4,4,41349,54095,0,6
2,Right Rack - Upper & Mid Tiers (Magazines),5,4,4,38709,62510,2,2
3,Right Rack - Bottom Tier (Newspapers),9,8,8,65010,175015,0,7
4,Foreground Right Endcap (Confectionery),2,0,0,99,0,0,0



2. SHELF_SIGNAL_METRIC — Scenario 2 (Facial & Postural Signals per Shelf):


,shelf_name,signal_type,signal_count
0,Left Chiller (Juices & Soft Drinks),INTERACTING / PRODUCT PICKUP,2
1,Left Chiller (Juices & Soft Drinks),FROWN,3
2,Left Chiller (Juices & Soft Drinks),NEUTRAL,5
3,Left Chiller (Juices & Soft Drinks),WALKING & BROWSING SHELF,4
4,Left Chiller (Juices & Soft Drinks),STANDING & ANALYZING SHELF,3
5,Left Chiller (Juices & Soft Drinks),INSPECTING / HESITATION,2
6,Left Chiller (Juices & Soft Drinks),SURPRISE,2
7,Left Chiller (Juices & Soft Drinks),SMILE,2
8,Left Chiller (Juices & Soft Drinks),HESITATION,2
9,Floor Multipack Stack (Bottled Water),NEUTRAL,6



3. Sample Payload (Scenario 2) -> frame_observation.attributes:
{
  "bbox": {
    "x1": 0.0,
    "y1": 0.3206,
    "x2": 0.0833,
    "y2": 0.9778,
    "confidence": 0.61
  },
  "pose": {
    "nose": [
      0.0291,
      0.4092
    ],
    "left_shoulder": [
      0.0098,
      0.4876
    ],
    "right_shoulder": [
      0.0,
      0.5033
    ],
    "left_wrist": [
      0.0425,
      0.6169
    ],
    "right_wrist": [
      0.0414,
      0.6294
    ],
    "left_ankle": [
      0.0408,
      0.9913
    ],
    "right_ankle": [
      0.0232,
      0.9996
    ]
  },
  "head_pose": {
    "yaw_estimated": 7.1,
    "pitch_estimated": -12.0,
    "roll_estimated": 0.0,
    "method": "3d_back_yaw",
    "quality": "medium"
  },
  "attention": {
    "direction_detected": true,
    "focus_duration_ms": 99,
    "shelf_confidence": 0.88
  },
  "hands": {
    "left": {
      "x": 0.0425,
      "y": 0.6169,
      "in_product_roi": true
    },
    "right": {
      "x": 0.0414,
      "y": 0.6294,
      

In [18]:
df_analytics_2 = pd.DataFrame(pipeline_iprox.flat_analytics_records)

# 1. Action-Segregated Spatial Heatmaps on Frame(0) for Scenario 2
actions_2 = df_analytics_2["body_action"].value_counts().index.tolist()
fig, axes = plt.subplots(1, len(actions_2), figsize=(5.5 * len(actions_2), 5.2))
if len(actions_2) == 1:
    axes = [axes]

for idx, act in enumerate(actions_2):
    df_act = df_analytics_2[df_analytics_2["body_action"] == act]
    hm_img = render_gaussian_heatmap(
        pipeline_iprox.frame_zero, df_act, col_x="gaze_x", col_y="gaze_y",
        shelves_geom=pipeline_iprox.shelves_geom, weight_col="engagement_score"
    )
    axes[idx].imshow(cv2.cvtColor(hm_img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f"{act}\n({len(df_act)} frames)", fontsize=10, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Scenario 2: Frame(0) Spatial Attention Heatmaps Segregated by Biomechanical Action", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# 2. Biomechanical Telemetry Time-Series for Primary Shopper in Scenario 2
top_shopper_2 = df_analytics_2["person_id"].value_counts().index[0]
df_shopper_2 = df_analytics_2[df_analytics_2["person_id"] == top_shopper_2].sort_values("timestamp_s")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df_shopper_2["timestamp_s"], df_shopper_2["theta_elbow_min"], label="Min Elbow Angle (deg)", color="#2980b9", lw=2)
axes[0].plot(df_shopper_2["timestamp_s"], df_shopper_2["theta_shoulder_max"], label="Max Shoulder Elevation (deg)", color="#e67e22", lw=2)
axes[0].axhline(115, color="#2980b9", linestyle="--", alpha=0.5, label="Inspection Threshold (<115 deg)")
axes[0].axhline(38, color="#e67e22", linestyle="--", alpha=0.5, label="Reach Threshold (>38 deg)")
axes[0].set_ylabel("Joint Angle (deg)")
axes[0].set_title(f"Scenario 2 Biomechanical Telemetry — Shopper {top_shopper_2} (Magazine Inspection & Reach)", fontsize=12, fontweight="bold")
axes[0].legend(loc="upper right", ncol=2)
axes[0].grid(True, linestyle=":", alpha=0.6)

axes[1].plot(df_shopper_2["timestamp_s"], df_shopper_2["r_reach_norm"], label="Reach Ratio (R_reach / Ws)", color="#27ae60", lw=2)
axes[1].plot(df_shopper_2["timestamp_s"], df_shopper_2["i_bimanual"], label="Bimanual Convergence (I_bimanual / Ws)", color="#8e44ad", lw=2)
axes[1].plot(df_shopper_2["timestamp_s"], df_shopper_2["i_head_adaptor"], label="Hand-to-Head Proximity (I_head_adaptor / Ws)", color="#e74c3c", lw=1.8, linestyle="--")
axes[1].axhline(0.85, color="#8e44ad", linestyle=":", alpha=0.6, label="Bimanual Coupling (<0.85 Ws)")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Dimensionless Ratio (Ws units)")
axes[1].legend(loc="upper right", ncol=2)
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### Published POC results: Aisle close-up

These figures preserve the completed demonstration run. Rerunning the preceding cell recomputes the plots from the current in-memory records.

![Action-grouped spatial heatmaps](../docs/media/aisle-closeup-heatmaps.png)

![Biomechanical telemetry time series](../docs/media/aisle-closeup-kinematics.png)

Heatmap intensities are normalized separately in each panel, and counts represent track-frame observations. These heuristic outputs have not been validated against ground truth.
